# Multinomial NB
- λειτουργεί πολύ καλά σε text classification
- model assumption ταιριάζει με τα δεδομένα

In [6]:
import pandas as pd
import numpy as np
from sklearn.naive_bayes import MultinomialNB
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test = pd.read_csv('test.csv')

def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    
    text = text.lower() # Convert to lowercase
    text = re.sub(r'\d+', ' ', text) # Remove digits
    text = re.sub(r'[^a-z\s]', ' ', text) # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra whitespace
    return text

In [3]:
def official_st1_score(y_true_hazard, y_pred_hazard, y_true_product, y_pred_product, verbose=True):
    #Βήμα 1: macro-F1 για hazard σε όλα τα δείγματα
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)

    # Βήμα 2: βρίσκουμε τα δείγματα όπου το hazard ήταν σωστό
    y_true_hazard = np.array(y_true_hazard)
    y_pred_hazard = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)

    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()

    # Βήμα 3: macro-F1 για product ΜΟΝΟ στα σωστά hazard
    if n_correct ==0:
        f1_product = 0.0
    else:
        f1_product = f1_score(y_true_product[correct_hazard_mask], y_pred_product[correct_hazard_mask], average='macro', zero_division=0)


    # Βήμα 4: official score
    score = (f1_hazard + f1_product) / 2

    if verbose:
        print(f'macro-F1 Hazard:   {f1_hazard:.4f}')
        print(f'Correct hazard predictions:  {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'----------------')
        print(f'OFFICIAL ST1 SCORE:   {score:.4f}')
    return score

In [ ]:
train['combined'] = train['title'].apply(preprocess_text) + ' ' + train['text'].str[:550].apply(preprocess_text)
valid['combined'] = valid['title'].apply(preprocess_text) + ' ' + valid['text'].str[:550].apply(preprocess_text)
test['combined'] = test['title'].apply(preprocess_text) + ' ' + test['text'].str[:550].apply(preprocess_text)

tfidf = TfidfVectorizer(max_features=5000,
                        ngram_range=(1,2),
                        sublinear_tf=True,
                        stop_words='english')

X_train = tfidf.fit_transform(train['combined'])
X_valid = tfidf.transform(valid['combined'])
X_test = tfidf.transform(test['combined'])

y_train_hazard = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard = valid['hazard-category']
y_valid_product = valid['product-category']

clf_hazard = MultinomialNB(
    alpha=1.0,
    fit_prior=True
)
clf_hazard.fit(X_train, y_train_hazard)

clf_product = MultinomialNB(
    alpha=1.0,
    fit_prior=True
)
clf_product.fit(X_train, y_train_product)

pred_hazard = clf_hazard.predict(X_valid)
pred_product = clf_product.predict(X_valid)

print('=== EVALUATION ===\n')
official_st1_score(
    valid['hazard-category'], pred_hazard,
    valid['product-category'], pred_product
)

pred_hazard_test  = clf_hazard.predict(X_test)
pred_product_test = clf_product.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_hazard_test,
    'product-category': pred_product_test
})

submission.to_csv('submission_phase24.csv', index=False)
print('\nSubmission file saved: submission_phase24.csv')
print(f'Αριθμός predictions: {len(submission)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission.head())

=== EVALUATION ===

macro-F1 Hazard:   0.4171
Correct hazard predictions:  474/565 (83.9%)
macro-F1 Product (given correct h): 0.2074
----------------
OFFICIAL ST1 SCORE:   0.3122

Submission file saved: submission_phase24.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological  meat, egg and dairy products
3   3      biological  meat, egg and dairy products
4   4  foreign bodies   cereals and bakery products


# Random Forest

In [13]:
clf_hazard_rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=40,
    max_features='sqrt',
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    criterion='entropy',
    bootstrap=True
)
clf_hazard_rf.fit(X_train, y_train_hazard)


clf_product_rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    max_depth=40,
    max_features='sqrt',
    min_samples_split=10,
    min_samples_leaf=4,
    class_weight="balanced",
    criterion='entropy',
    bootstrap=True
)
clf_product_rf.fit(X_train, y_train_product)

pred_hazard_rf = clf_hazard_rf.predict(X_valid)
pred_product_rf = clf_product_rf.predict(X_valid)

print('\n=== Rating RF ===\n')
score_rf = official_st1_score(
    valid['hazard-category'], pred_hazard_rf,
    valid['product-category'], pred_product_rf
)

print(f'\n=== Evaluation ===')
print(f' Random Forest                ( title+text[:550]): {score_rf:.4f}')


pred_hazard_test_rf  = clf_hazard_rf.predict(X_test)
pred_product_test_rf = clf_product_rf.predict(X_test)

submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': pred_hazard_test_rf,
    'product-category': pred_product_test_rf
})

submission.to_csv('submission_phase24_rf.csv', index=False)
print('\nSubmission file saved: submission_phase24_rf.csv')
print(f'Αριθμός predictions: {len(submission)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission.head())


=== Rating RF ===

macro-F1 Hazard:   0.7199
Correct hazard predictions:  499/565 (88.3%)
macro-F1 Product (given correct h): 0.2996
----------------
OFFICIAL ST1 SCORE:   0.5097

=== Evaluation ===
 Random Forest                ( title+text[:550]): 0.5097

Submission file saved: submission_phase24_rf.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological  meat, egg and dairy products
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts
